# 証券営業インテリジェンス ハンズオン
## Part 3: Cortex Analyst（自然言語 to SQL）

このノートブックでは、**Cortex Analyst** のセットアップと動作確認を行います。

### Cortex Analyst とは

日本語の質問を入力するだけで、Snowflakeが**自動的にSQLを生成・実行**してくれる機能です。

```
ユーザー: 「資産5億円以上で相続対策に関心がある顧客は何人いますか？」
                    ↓
Cortex Analyst  →  SELECT COUNT(*) FROM DIM_CUSTOMER WHERE ...
                    ↓
          データベースから自動的に回答を取得
```

### 実装方法: Semantic View

Cortex Analyst は **Semantic View（セマンティックビュー）** というメタデータ定義ファイルを使用します。

- テーブルの意味・関係性・同義語を定義
- 「顧客数」「AUM」などビジネス用語を認識
- 複数テーブルの結合ロジックを自動処理

### 🚀 アハ体験ポイント

> **「SQLを知らない営業担当者が日本語で質問するだけで、顧客DB全体を分析できる」**
> 
> 例: 「今月コンタクトしていない資産3億円以上の顧客リストを出して」
> → Cortex Analyst が自動でSQLを生成・実行！

In [ ]:
%%sql -r result_use
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE COMPUTE_WH;
SELECT 'Ready!' AS STATUS;

## Step 1: Semantic View の作成

Semantic View を作成して、Cortex Analyst が理解できるデータモデルを定義します。

**顧客資産管理 Semantic View** に含まれるもの:
- テーブル定義と関係性（JOIN条件）
- ビジネス指標（KPI）: 顧客数、AUM（資産合計）、取引金額等
- 日本語の同義語: 「顧客」= 「お客様」= 「クライアント」

In [ ]:
%%sql -r result_sv1
-- ============================================================
-- Semantic View 1: 顧客資産管理（Cortex Analyst 1）
-- ============================================================
CREATE OR REPLACE SEMANTIC VIEW SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW

TABLES (
    CUSTOMERS AS SNOWFINANCE_DB.DEMO_SCHEMA.DIM_CUSTOMER
        PRIMARY KEY (CUSTOMER_ID)
        WITH SYNONYMS = ('顧客', 'お客様', 'クライアント', '顧客マスタ')
        COMMENT = '顧客マスタテーブル',

    FAMILY AS SNOWFINANCE_DB.DEMO_SCHEMA.DIM_FAMILY
        PRIMARY KEY (FAMILY_ID)
        WITH SYNONYMS = ('家族', '親族', '相続人', '家族情報')
        COMMENT = '家族構成テーブル',

    LIFE_EVENTS AS SNOWFINANCE_DB.DEMO_SCHEMA.DIM_LIFE_EVENT
        PRIMARY KEY (EVENT_ID)
        WITH SYNONYMS = ('ライフイベント', 'イベント', '予定', '計画')
        COMMENT = 'ライフイベントテーブル',

    PORTFOLIO AS SNOWFINANCE_DB.DEMO_SCHEMA.FACT_PORTFOLIO
        PRIMARY KEY (PORTFOLIO_ID)
        WITH SYNONYMS = ('ポートフォリオ', '保有資産', '持ち株', '保有銘柄')
        COMMENT = 'ポートフォリオテーブル',

    TRANSACTIONS AS SNOWFINANCE_DB.DEMO_SCHEMA.FACT_TRANSACTION
        PRIMARY KEY (TRANSACTION_ID)
        WITH SYNONYMS = ('取引', '売買', 'トランザクション', '取引履歴')
        COMMENT = '取引履歴テーブル',

    INHERITANCE_TAX AS SNOWFINANCE_DB.DEMO_SCHEMA.V_INHERITANCE_TAX_ESTIMATE
        PRIMARY KEY (CUSTOMER_ID)
        WITH SYNONYMS = ('相続税', '相続税試算', '相続')
        COMMENT = '相続税試算ビュー'
)

RELATIONSHIPS (
    CUSTOMERS_TO_FAMILY       AS CUSTOMERS    (CUSTOMER_ID) REFERENCES FAMILY,
    CUSTOMERS_TO_LIFE_EVENTS  AS CUSTOMERS    (CUSTOMER_ID) REFERENCES LIFE_EVENTS,
    CUSTOMERS_TO_PORTFOLIO    AS CUSTOMERS    (CUSTOMER_ID) REFERENCES PORTFOLIO,
    CUSTOMERS_TO_TRANSACTIONS AS CUSTOMERS    (CUSTOMER_ID) REFERENCES TRANSACTIONS,
    CUSTOMERS_TO_INHERITANCE  AS CUSTOMERS    (CUSTOMER_ID) REFERENCES INHERITANCE_TAX
)

FACTS (
    CUSTOMERS.CUSTOMER_RECORD   AS 1              COMMENT = '顧客レコード数',
    CUSTOMERS.TOTAL_ASSETS      AS TOTAL_ASSETS   COMMENT = '総資産（円）',
    CUSTOMERS.LIQUID_ASSETS     AS LIQUID_ASSETS  COMMENT = '流動資産（円）',
    PORTFOLIO.PORTFOLIO_RECORD  AS 1              COMMENT = 'ポートフォリオレコード数',
    PORTFOLIO.MARKET_VALUE      AS MARKET_VALUE   COMMENT = '時価評価額（円）',
    PORTFOLIO.UNREALIZED_GAIN   AS UNREALIZED_GAIN COMMENT = '含み益（円）',
    PORTFOLIO.QUANTITY          AS QUANTITY       COMMENT = '保有数量',
    TRANSACTIONS.TRANSACTION_RECORD AS 1          COMMENT = '取引レコード数',
    TRANSACTIONS.AMOUNT         AS AMOUNT         COMMENT = '取引金額（円）',
    LIFE_EVENTS.LIFE_EVENT_RECORD AS 1            COMMENT = 'ライフイベントレコード数',
    LIFE_EVENTS.ESTIMATED_AMOUNT AS ESTIMATED_AMOUNT COMMENT = 'ライフイベント必要金額（円）',
    INHERITANCE_TAX.ESTIMATED_TAX AS ESTIMATED_TAX COMMENT = '推定相続税額（円）'
)

DIMENSIONS (
    CUSTOMERS.CUSTOMER_ID       AS CUSTOMER_ID       WITH SYNONYMS = ('顧客ID', 'ID') COMMENT = '顧客識別子',
    CUSTOMERS.CUSTOMER_NAME     AS CUSTOMER_NAME     WITH SYNONYMS = ('顧客名', '氏名', '名前') COMMENT = '顧客名',
    CUSTOMERS.AGE               AS AGE               WITH SYNONYMS = ('年齢') COMMENT = '年齢',
    CUSTOMERS.GENDER            AS GENDER            WITH SYNONYMS = ('性別') COMMENT = '性別',
    CUSTOMERS.OCCUPATION        AS OCCUPATION        WITH SYNONYMS = ('職業') COMMENT = '職業',
    CUSTOMERS.PREFECTURE        AS PREFECTURE        WITH SYNONYMS = ('都道府県', '居住地', '住所') COMMENT = '居住都道府県',
    CUSTOMERS.ANNUAL_INCOME_BAND AS ANNUAL_INCOME_BAND WITH SYNONYMS = ('年収', '年収帯') COMMENT = '年収帯',
    CUSTOMERS.RISK_TOLERANCE    AS RISK_TOLERANCE    WITH SYNONYMS = ('リスク許容度', 'リスク') COMMENT = 'リスク許容度',
    CUSTOMERS.INVESTMENT_PURPOSE AS INVESTMENT_PURPOSE WITH SYNONYMS = ('投資目的') COMMENT = '投資目的',
    CUSTOMERS.SEGMENT           AS SEGMENT           WITH SYNONYMS = ('セグメント', '顧客区分', 'ランク') COMMENT = 'セグメント',
    CUSTOMERS.RM_ID             AS RM_ID             WITH SYNONYMS = ('担当者', 'RM') COMMENT = '担当RM ID',
    CUSTOMERS.RM_NAME           AS RM_NAME           WITH SYNONYMS = ('担当者名', 'RM名') COMMENT = '担当RM名',
    FAMILY.RELATIONSHIP         AS RELATIONSHIP      WITH SYNONYMS = ('続柄', '家族関係') COMMENT = '続柄',
    FAMILY.IS_HEIR              AS IS_HEIR           WITH SYNONYMS = ('相続人', '相続人フラグ') COMMENT = '相続人フラグ',
    LIFE_EVENTS.EVENT_TYPE      AS EVENT_TYPE        WITH SYNONYMS = ('イベント種別', '相談内容') COMMENT = 'ライフイベント種別',
    LIFE_EVENTS.URGENCY         AS URGENCY           WITH SYNONYMS = ('緊急度', '優先度') COMMENT = '緊急度',
    PORTFOLIO.ASSET_CLASS       AS ASSET_CLASS       WITH SYNONYMS = ('資産クラス', '種別') COMMENT = '資産クラス',
    PORTFOLIO.SECURITY_NAME     AS SECURITY_NAME     WITH SYNONYMS = ('銘柄名', '銘柄') COMMENT = '銘柄名',
    PORTFOLIO.SECURITY_CODE     AS SECURITY_CODE     WITH SYNONYMS = ('銘柄コード', '証券コード') COMMENT = '銘柄コード',
    PORTFOLIO.ACCOUNT_TYPE      AS ACCOUNT_TYPE      WITH SYNONYMS = ('口座種別', '口座') COMMENT = '口座種別',
    TRANSACTIONS.TRANSACTION_DATE AS TRANSACTION_DATE WITH SYNONYMS = ('取引日', '売買日') COMMENT = '取引日',
    TRANSACTIONS.TRANSACTION_TYPE AS TRANSACTION_TYPE WITH SYNONYMS = ('取引種別', '売買種別') COMMENT = '取引種別'
)

METRICS (
    CUSTOMERS.TOTAL_CUSTOMERS     AS COUNT(CUSTOMERS.CUSTOMER_RECORD) WITH SYNONYMS = ('顧客数', '人数', '件数') COMMENT = '顧客総数',
    CUSTOMERS.AVG_TOTAL_ASSETS    AS AVG(CUSTOMERS.TOTAL_ASSETS)      WITH SYNONYMS = ('平均資産', '平均預かり資産') COMMENT = '平均総資産',
    CUSTOMERS.SUM_TOTAL_ASSETS    AS SUM(CUSTOMERS.TOTAL_ASSETS)      WITH SYNONYMS = ('総資産合計', 'AUM', '預かり資産合計') COMMENT = '預かり資産合計',
    PORTFOLIO.TOTAL_MARKET_VALUE  AS SUM(PORTFOLIO.MARKET_VALUE)      WITH SYNONYMS = ('評価額合計', '時価総額') COMMENT = '保有資産評価額合計',
    PORTFOLIO.TOTAL_UNREALIZED_GAIN AS SUM(PORTFOLIO.UNREALIZED_GAIN) WITH SYNONYMS = ('含み益合計', '含み損益合計') COMMENT = '含み益合計',
    TRANSACTIONS.TOTAL_TRANSACTIONS AS COUNT(TRANSACTIONS.TRANSACTION_RECORD) WITH SYNONYMS = ('取引件数', '売買件数') COMMENT = '取引件数合計',
    TRANSACTIONS.TOTAL_TRANSACTION_AMOUNT AS SUM(TRANSACTIONS.AMOUNT) WITH SYNONYMS = ('取引金額合計', '売買代金') COMMENT = '取引金額合計',
    INHERITANCE_TAX.TOTAL_INHERITANCE_TAX AS SUM(INHERITANCE_TAX.ESTIMATED_TAX) WITH SYNONYMS = ('相続税合計', '潜在相続税') COMMENT = '推定相続税額合計'
)

COMMENT = '顧客資産管理セマンティックビュー。顧客情報・ポートフォリオ・取引履歴・ライフイベント・相続税試算を統合分析。';

SELECT 'Customer Wealth Semantic View created!' AS STATUS;

In [ ]:
%%sql -r result_sv2
-- ============================================================
-- Semantic View 2: 信託銀行商品（Cortex Analyst 2）
-- ============================================================
CREATE OR REPLACE SEMANTIC VIEW SNOWFINANCE_DB.DEMO_SCHEMA.TRUST_PRODUCT_SEMANTIC_VIEW

TABLES (
    PRODUCTS AS SNOWFINANCE_DB.DEMO_SCHEMA.DIM_TRUST_PRODUCT
        PRIMARY KEY (PRODUCT_ID)
        WITH SYNONYMS = ('商品', '信託商品', 'ローン', '金融商品')
        COMMENT = '信託銀行の商品マスタ',

    RECOMMENDATIONS AS SNOWFINANCE_DB.DEMO_SCHEMA.DIM_PRODUCT_RECOMMENDATION
        PRIMARY KEY (RECOMMENDATION_ID)
        WITH SYNONYMS = ('推奨', 'レコメンド', '提案', '推薦')
        COMMENT = '商品推奨ロジック'
)

RELATIONSHIPS (
    RECOMMENDATIONS (PRODUCT_ID) REFERENCES PRODUCTS
)

FACTS (
    PRODUCTS.MIN_LOAN_AMOUNT   AS MIN_AMOUNT          WITH SYNONYMS = ('最低金額', '最小融資額') COMMENT = '最低融資・信託金額',
    PRODUCTS.MAX_LOAN_AMOUNT   AS MAX_AMOUNT          WITH SYNONYMS = ('最高金額', '最大融資額') COMMENT = '最高融資・信託金額',
    PRODUCTS.MIN_INTEREST_RATE AS INTEREST_RATE_MIN   WITH SYNONYMS = ('最低金利', '金利下限') COMMENT = '最低金利（%）',
    PRODUCTS.MAX_INTEREST_RATE AS INTEREST_RATE_MAX   WITH SYNONYMS = ('最高金利', '金利上限') COMMENT = '最高金利（%）',
    RECOMMENDATIONS.RECOMMENDATION_PRIORITY AS PRIORITY WITH SYNONYMS = ('優先度', '推奨度') COMMENT = '推奨優先度'
)

DIMENSIONS (
    PRODUCTS.PRODUCT_ID          AS PRODUCT_ID         WITH SYNONYMS = ('商品ID') COMMENT = '商品識別子',
    PRODUCTS.PRODUCT_NAME        AS PRODUCT_NAME       WITH SYNONYMS = ('商品名', '名称') COMMENT = '商品名',
    PRODUCTS.PRODUCT_CATEGORY    AS PRODUCT_CATEGORY   WITH SYNONYMS = ('商品カテゴリ', '種別', 'カテゴリ') COMMENT = '商品カテゴリ',
    PRODUCTS.DESCRIPTION         AS DESCRIPTION        WITH SYNONYMS = ('商品説明', '概要') COMMENT = '商品説明',
    PRODUCTS.ELIGIBLE_SEGMENT    AS ELIGIBLE_SEGMENT   WITH SYNONYMS = ('対象セグメント', '対象顧客') COMMENT = '対象顧客セグメント',
    PRODUCTS.TAX_BENEFIT         AS TAX_BENEFIT        WITH SYNONYMS = ('税制メリット', '節税効果', '税金優遇') COMMENT = '税制上のメリット',
    PRODUCTS.RISKS               AS RISKS              WITH SYNONYMS = ('リスク', '注意点') COMMENT = 'リスク',
    RECOMMENDATIONS.TARGET_LIFE_EVENT AS TARGET_LIFE_EVENT WITH SYNONYMS = ('対象イベント', '提案シーン') COMMENT = '推奨対象ライフイベント',
    RECOMMENDATIONS.RECOMMENDATION_REASON AS RECOMMENDATION_REASON WITH SYNONYMS = ('推奨理由', '提案理由') COMMENT = '推奨理由',
    RECOMMENDATIONS.BENEFIT_DESCRIPTION AS BENEFIT_DESCRIPTION WITH SYNONYMS = ('メリット', '効果') COMMENT = 'メリット説明',
    RECOMMENDATIONS.COMPARISON_WITH_ALTERNATIVE AS COMPARISON_WITH_ALTERNATIVE WITH SYNONYMS = ('比較', '代替案') COMMENT = '代替案との比較'
)

METRICS (
    PRODUCTS.PRODUCT_COUNT AS COUNT(PRODUCTS.PRODUCT_ID) WITH SYNONYMS = ('商品数') COMMENT = '商品数',
    RECOMMENDATIONS.RECOMMENDATION_COUNT AS COUNT(RECOMMENDATIONS.RECOMMENDATION_ID) WITH SYNONYMS = ('推奨パターン数') COMMENT = '推奨パターン数'
)

COMMENT = '信託銀行商品セマンティックビュー。ローン・信託商品の情報と推奨ロジックを統合。';

SELECT 'Trust Product Semantic View created!' AS STATUS;

In [ ]:
%%sql -r result_verify_sv
SHOW SEMANTIC VIEWS IN SCHEMA DEMO_SCHEMA;

## Step 2: Cortex Analyst のテスト

Python から Cortex Analyst REST API を呼び出してみましょう。

> **Note**: Snowflake Notebook では `snowflake.cortex` モジュールから直接呼び出せます。
> REST API 経由での呼び出し例も後述します。

### テスト質問例

1. **「資産5億円以上の顧客を教えて」**
2. **「トヨタ株を保有している顧客は何人いますか？」**
3. **「教育資金のライフイベントがある顧客の一覧を出してください」**
4. **「担当RMごとの預かり資産合計を教えて」**

In [ ]:
import json
import _snowflake
import streamlit as st

# Cortex Analyst APIを呼び出す関数
def ask_cortex_analyst(question: str, semantic_view: str) -> dict:
    request_body = {
        "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
        "semantic_model_file": f"@{semantic_view}"  # Semantic View を使用
    }
    
    # セマンティックビューを直接指定
    response = _snowflake.send_snow_api_request(
        "POST",
        "/api/v2/cortex/analyst/message",
        {},
        {},
        {
            "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
            "semantic_view": "SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW"
        },
        None,
        30000
    )
    return json.loads(response["content"])

# テスト質問
test_questions = [
    "資産5億円以上の顧客数を教えてください",
    "トヨタ株（7203）を保有している顧客は何人いますか？",
    "教育資金のライフイベントがある顧客の一覧を出してください",
]

print("Cortex Analyst テスト質問:")
for i, q in enumerate(test_questions, 1):
    print(f"  {i}. {q}")
print()
print("NOTE: 実際の実行は Snowflake Notebook 上で行ってください。")
print("      REST API エンドポイント: /api/v2/cortex/analyst/message")

## Step 3: Snowsight Cortex Analyst UI での確認

Semantic View が正しく作成されたら、Snowsight の Cortex Analyst UI で自然言語クエリを試しましょう。

**アクセス方法:**
1. Snowsight にログイン
2. 左メニュー「AI & ML」→「Cortex Analyst」
3. Semantic View に `SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW` を選択

**試してみる質問:**

```
Q1: 資産5億円以上の顧客をセグメント別に教えてください

Q2: トヨタ株（銘柄コード7203）を保有している顧客は何人いますか？
    その合計含み益も教えてください。

Q3: 教育資金のライフイベントがあり、緊急度が高い顧客の
    氏名と必要金額を一覧で出してください

Q4: 担当RMごとの預かり資産合計（AUM）を降順で教えてください

Q5: 相続税が1億円を超える可能性がある顧客は何人いますか？
    年齢帯別に集計してください

Q6: C001の山田太郎様の詳細プロフィールとポートフォリオを教えてください
```

In [ ]:
%%sql -r result_manual_test
-- Cortex Analyst が生成するクエリに近いSQLを手動で実行してみる
-- Q: 「資産5億円以上の顧客数をセグメント別に教えてください」
SELECT
    SEGMENT     AS セグメント,
    COUNT(*)    AS 顧客数,
    ROUND(AVG(TOTAL_ASSETS)/100000000, 2) AS 平均資産_億円,
    ROUND(SUM(TOTAL_ASSETS)/100000000, 0) AS 資産合計_億円
FROM DIM_CUSTOMER
WHERE TOTAL_ASSETS >= 500000000
GROUP BY SEGMENT
ORDER BY 資産合計_億円 DESC;

In [ ]:
%%sql -r result_manual_test2
-- Q: 「教育資金のライフイベントがあり、緊急度が高い顧客一覧」
SELECT
    c.CUSTOMER_ID,
    c.CUSTOMER_NAME,
    c.AGE,
    c.SEGMENT,
    e.EVENT_TYPE,
    ROUND(e.ESTIMATED_AMOUNT/10000, 0) AS 必要金額_万円,
    e.URGENCY,
    c.RM_NAME AS 担当RM
FROM DIM_CUSTOMER c
JOIN DIM_LIFE_EVENT e ON c.CUSTOMER_ID = e.CUSTOMER_ID
WHERE e.EVENT_TYPE = '教育資金'
  AND e.URGENCY = '高'
ORDER BY e.ESTIMATED_AMOUNT DESC;

## まとめ

Part 3 完了！Cortex Analyst のセットアップが完了しました。

### 作成したオブジェクト

| Semantic View | 対象 | テーブル数 |
|---|---|---|
| CUSTOMER_WEALTH_SEMANTIC_VIEW | 顧客資産管理 | 6テーブル/ビュー |
| TRUST_PRODUCT_SEMANTIC_VIEW | 信託銀行商品 | 2テーブル |

### Cortex Analyst の強み

- **SQLゼロ**: 自然言語だけで複雑な多テーブル結合クエリを生成
- **ビジネス用語対応**: 「AUM」「相続人」「含み益」などの業界用語を理解
- **日本語サポート**: 日本語の質問・同義語に対応

**次のステップ:** `04_cortex_search.ipynb` でセマンティック検索を体験してください。